# Data Lake Fundamentals with PySpark and AWS S3

## Class Notebook: Building a Realistic E-Commerce Data Lake

In this notebook, we will build a practical Data Lake using these source systems:

```text
orders.csv
customers.csv
products.json
payments.csv
clickstream_events.json
```

## What you will build

```text
Local Data Lake
    bronze/
    silver/
    gold/

Optional AWS S3 Data Lake
    s3://your-bucket/ecommerce-datalake/bronze/
    s3://your-bucket/ecommerce-datalake/silver/
    s3://your-bucket/ecommerce-datalake/gold/
```

## Learning objectives

By the end of this session, you should be able to explain:

- What a Data Lake is
- Why Data Lakes are used in Data Engineering
- Bronze, Silver, and Gold layers
- How CSV, JSON, and Parquet fit into a Data Lake
- How PySpark processes files in a Data Lake
- How AWS S3 can be used as a Data Lake storage layer
- Why this topic prepares us for Delta Lake and Databricks


# 1. Big Picture: What is a Data Lake?

A Data Lake is a centralized storage layer where companies store raw and processed data at scale.

Unlike a traditional database, a Data Lake can store:

```text
Structured data       -> CSV, relational exports
Semi-structured data  -> JSON, API responses, logs
Unstructured data     -> images, documents, audio, video
```

## Simple definition

```text
A Data Lake is a scalable storage area where raw and processed data files are organized for analytics, machine learning, and downstream data pipelines.
```


# 2. Overall Data Lake Architecture

```mermaid
flowchart LR
    A[Source Systems] --> B[Bronze Layer<br/>Raw Data]
    B --> C[Silver Layer<br/>Cleaned and Standardized]
    C --> D[Gold Layer<br/>Business Ready Analytics]
    D --> E[BI Dashboards]
    D --> F[Data Science]
    D --> G[Reporting Tables]

    A1[orders.csv] --> A
    A2[customers.csv] --> A
    A3[products.json] --> A
    A4[payments.csv] --> A
    A5[clickstream_events.json] --> A
```

## One-line summary

```text
Bronze stores raw data.
Silver stores cleaned data.
Gold stores business-ready data.
```


# 3. Data Lake vs Database vs Data Warehouse

| System | Purpose | Example |
|---|---|---|
| Database | Runs an application | MySQL/PostgreSQL for an order app |
| Data Warehouse | Business reporting and analytics | Snowflake/Redshift/BigQuery |
| Data Lake | Stores large raw and processed files | S3/ADLS/GCS/HDFS |
| Lakehouse | Reliable tables on top of a Data Lake | Delta Lake/Databricks |

## Teaching connection

```text
Databases generate source data.
Data Lakes store raw and processed files.
PySpark transforms Data Lake files.
Warehouses or Lakehouses serve business analytics.
```


# 4. Bronze, Silver, and Gold Layers

```mermaid
flowchart TD
    S[Source Files<br/>CSV + JSON] --> B[Bronze<br/>Raw as received]
    B --> S1[Silver<br/>Cleaned and validated]
    S1 --> G[Gold<br/>Aggregated business datasets]

    B --> BNote[Keep original data<br/>Minimal transformation]
    S1 --> SNote[Fix schema<br/>Remove duplicates<br/>Validate records]
    G --> GNote[Revenue reports<br/>Customer metrics<br/>Product analytics]
```

## Bronze Layer

Stores the original data as received.

Example:

```text
bronze/orders/ingestion_date=2026-08-10/orders.csv
bronze/products/ingestion_date=2026-08-10/products.json
```

## Silver Layer

Stores cleaned and standardized data.

Example:

```text
silver/orders/order_date=2026-08-10/
silver/customers/
```

## Gold Layer

Stores business-ready output.

Example:

```text
gold/revenue_by_city/
gold/product_performance/
gold/customer_summary/
```


# 5. Setup PySpark

This notebook is designed for Google Colab.

Run the next cell to install PySpark.


In [ ]:
!pip install pyspark -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lower, trim, to_date, current_timestamp,
    sum as spark_sum, count, avg, when, lit, desc
)

spark = (
    SparkSession.builder
    .appName("DataLakeFundamentalsEcommerce")
    .getOrCreate()
)

spark


# 6. Create Source Data

We will create five source files:

```text
orders.csv
customers.csv
products.json
payments.csv
clickstream_events.json
```

These represent data from multiple systems inside an e-commerce company.

```mermaid
flowchart LR
    O[Order Management System<br/>orders.csv] --> DL[Data Lake]
    C[CRM System<br/>customers.csv] --> DL
    P[Product Catalog API<br/>products.json] --> DL
    Pay[Payment Gateway<br/>payments.csv] --> DL
    Click[Website/App Events<br/>clickstream_events.json] --> DL
```


In [ ]:
from pathlib import Path
import json
import csv
import shutil

base_path = Path("ecommerce_datalake_demo")

# Clean previous run
if base_path.exists():
    shutil.rmtree(base_path)

source_path = base_path / "source_files"
source_path.mkdir(parents=True, exist_ok=True)

source_path


PosixPath('ecommerce_datalake_demo/source_files')

## 6.1 Create orders.csv

This file intentionally includes real-world issues:

```text
duplicate order_id
negative amount
missing customer_id
mixed case status
invalid amount
```


In [ ]:
orders_rows = [
    ["order_id", "customer_id", "product_id", "order_date", "amount", "status", "city"],
    [1001, "C001", "P001", "2026-08-10", 2500, "Completed", "Delhi"],
    [1002, "C002", "P002", "2026-08-10", -500, "completed", "Mumbai"],
    [1003, "C003", "P003", "2026-08-10", "", "Pending", "Delhi"],
    [1004, "C004", "P004", "2026-08-11", 3200, "Completed", "Pune"],
    [1005, "C005", "P001", "2026-08-11", 1200, "Cancelled", "Mumbai"],
    [1006, "", "P002", "2026-08-11", 1800, "Completed", "Chennai"],
    [1007, "C002", "P005", "2026-08-12", 4500, "completed", "Bangalore"],
    [1008, "C003", "P003", "2026-08-12", 900, "FAILED", "Delhi"],
    [1001, "C001", "P001", "2026-08-10", 2500, "Completed", "Delhi"],
]

orders_file = source_path / "orders.csv"

with open(orders_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(orders_rows)

print(orders_file)


ecommerce_datalake_demo/source_files/orders.csv


## 6.2 Create customers.csv


In [ ]:
customers_rows = [
    ["customer_id", "customer_name", "segment", "signup_date", "country"],
    ["C001", "Aarav Sharma", "Premium", "2025-01-15", "India"],
    ["C002", "Meera Iyer", "Standard", "2025-03-20", "India"],
    ["C003", "Kabir Khan", "Premium", "2025-05-01", "India"],
    ["C004", "Ananya Rao", "Standard", "2025-07-11", "India"],
    ["C005", "Rohan Gupta", "New", "2026-01-01", "India"],
]

customers_file = source_path / "customers.csv"

with open(customers_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(customers_rows)

print(customers_file)


ecommerce_datalake_demo/source_files/customers.csv


## 6.3 Create products.json

Products are often received from APIs in JSON format.


In [ ]:
products_data = [
    {"product_id": "P001", "product_name": "Laptop", "category": "Electronics", "price": 2500},
    {"product_id": "P002", "product_name": "Headphones", "category": "Electronics", "price": 1800},
    {"product_id": "P003", "product_name": "Backpack", "category": "Fashion", "price": 900},
    {"product_id": "P004", "product_name": "Office Chair", "category": "Furniture", "price": 3200},
    {"product_id": "P005", "product_name": "Smart Watch", "category": "Electronics", "price": 4500},
]

products_file = source_path / "products.json"

with open(products_file, "w") as f:
    for item in products_data:
        f.write(json.dumps(item) + "\n")

print(products_file)


ecommerce_datalake_demo/source_files/products.json


## 6.4 Create payments.csv


In [ ]:
payments_rows = [
    ["payment_id", "order_id", "payment_method", "payment_status", "payment_date"],
    ["PMT001", 1001, "UPI", "success", "2026-08-10"],
    ["PMT002", 1002, "card", "failed", "2026-08-10"],
    ["PMT003", 1004, "netbanking", "success", "2026-08-11"],
    ["PMT004", 1007, "UPI", "success", "2026-08-12"],
    ["PMT005", 1008, "card", "failed", "2026-08-12"],
]

payments_file = source_path / "payments.csv"

with open(payments_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(payments_rows)

print(payments_file)


ecommerce_datalake_demo/source_files/payments.csv


## 6.5 Create clickstream_events.json

Clickstream data usually comes from websites, apps, or event tracking systems.


In [ ]:
clickstream_data = [
    {"event_id": "E001", "customer_id": "C001", "event_time": "2026-08-10 10:01:00", "event_type": "view", "product_id": "P001"},
    {"event_id": "E002", "customer_id": "C001", "event_time": "2026-08-10 10:03:00", "event_type": "add_to_cart", "product_id": "P001"},
    {"event_id": "E003", "customer_id": "C001", "event_time": "2026-08-10 10:05:00", "event_type": "purchase", "product_id": "P001"},
    {"event_id": "E004", "customer_id": "C002", "event_time": "2026-08-10 11:00:00", "event_type": "view", "product_id": "P002"},
    {"event_id": "E005", "customer_id": "C003", "event_time": "2026-08-11 09:30:00", "event_type": "view", "product_id": "P003"},
    {"event_id": "E006", "customer_id": "C004", "event_time": "2026-08-11 12:10:00", "event_type": "purchase", "product_id": "P004"},
    {"event_id": "E007", "customer_id": "C002", "event_time": "2026-08-12 15:45:00", "event_type": "purchase", "product_id": "P005"},
]

clickstream_file = source_path / "clickstream_events.json"

with open(clickstream_file, "w") as f:
    for event in clickstream_data:
        f.write(json.dumps(event) + "\n")

print(clickstream_file)


ecommerce_datalake_demo/source_files/clickstream_events.json


# 7. Create Local Data Lake Folder Structure

We will create this structure locally first:

```text
ecommerce_datalake_demo/
  source_files/
  data_lake/
    bronze/
    silver/
    gold/
```

This simulates how a Data Lake would be organized on AWS S3.


In [ ]:
data_lake_path = base_path / "data_lake"

bronze_path = data_lake_path / "bronze"
silver_path = data_lake_path / "silver"
gold_path = data_lake_path / "gold"

for path in [bronze_path, silver_path, gold_path]:
    path.mkdir(parents=True, exist_ok=True)

print("Bronze:", bronze_path)
print("Silver:", silver_path)
print("Gold:", gold_path)


Bronze: ecommerce_datalake_demo/data_lake/bronze
Silver: ecommerce_datalake_demo/data_lake/silver
Gold: ecommerce_datalake_demo/data_lake/gold


# 8. Bronze Layer: Store Raw Data As Received

The Bronze layer stores the source files with minimal changes.

This is useful because:

```text
We preserve the original data
We can replay pipelines later
We can debug data quality issues
We keep traceability from source to analytics
```

```mermaid
flowchart LR
    S[Source Files] --> B[Bronze Layer]
    B --> O[orders.csv]
    B --> C[customers.csv]
    B --> P[products.json]
    B --> Pay[payments.csv]
    B --> E[clickstream_events.json]
```


In [ ]:
ingestion_date = "2026-08-10"

bronze_orders_path = bronze_path / "orders" / f"ingestion_date={ingestion_date}"
bronze_customers_path = bronze_path / "customers" / f"ingestion_date={ingestion_date}"
bronze_products_path = bronze_path / "products" / f"ingestion_date={ingestion_date}"
bronze_payments_path = bronze_path / "payments" / f"ingestion_date={ingestion_date}"
bronze_clickstream_path = bronze_path / "clickstream_events" / f"ingestion_date={ingestion_date}"

for path in [bronze_orders_path, bronze_customers_path, bronze_products_path, bronze_payments_path, bronze_clickstream_path]:
    path.mkdir(parents=True, exist_ok=True)

shutil.copy(orders_file, bronze_orders_path / "orders.csv")
shutil.copy(customers_file, bronze_customers_path / "customers.csv")
shutil.copy(products_file, bronze_products_path / "products.json")
shutil.copy(payments_file, bronze_payments_path / "payments.csv")
shutil.copy(clickstream_file, bronze_clickstream_path / "clickstream_events.json")

print("Bronze layer created successfully.")


Bronze layer created successfully.


In [ ]:
for file_path in bronze_path.rglob("*"):
    if file_path.is_file():
        print(file_path)


ecommerce_datalake_demo/data_lake/bronze/payments/ingestion_date=2026-08-10/payments.csv
ecommerce_datalake_demo/data_lake/bronze/clickstream_events/ingestion_date=2026-08-10/clickstream_events.json
ecommerce_datalake_demo/data_lake/bronze/orders/ingestion_date=2026-08-10/orders.csv
ecommerce_datalake_demo/data_lake/bronze/products/ingestion_date=2026-08-10/products.json
ecommerce_datalake_demo/data_lake/bronze/customers/ingestion_date=2026-08-10/customers.csv


# 9. Read Bronze Data with PySpark

Now we read raw data from the Bronze layer.

In real-world pipelines, PySpark jobs often read raw files from S3, ADLS, GCS, or HDFS.


In [ ]:
bronze_orders = spark.read.csv(str(bronze_orders_path / "orders.csv"), header=True, inferSchema=True)
bronze_customers = spark.read.csv(str(bronze_customers_path / "customers.csv"), header=True, inferSchema=True)
bronze_products = spark.read.json(str(bronze_products_path / "products.json"))
bronze_payments = spark.read.csv(str(bronze_payments_path / "payments.csv"), header=True, inferSchema=True)
bronze_clickstream = spark.read.json(str(bronze_clickstream_path / "clickstream_events.json"))

bronze_orders.show()


+--------+-----------+----------+----------+------+---------+---------+
|order_id|customer_id|product_id|order_date|amount|   status|     city|
+--------+-----------+----------+----------+------+---------+---------+
|    1001|       C001|      P001|2026-08-10|  2500|Completed|    Delhi|
|    1002|       C002|      P002|2026-08-10|  -500|completed|   Mumbai|
|    1003|       C003|      P003|2026-08-10|  NULL|  Pending|    Delhi|
|    1004|       C004|      P004|2026-08-11|  3200|Completed|     Pune|
|    1005|       C005|      P001|2026-08-11|  1200|Cancelled|   Mumbai|
|    1006|       NULL|      P002|2026-08-11|  1800|Completed|  Chennai|
|    1007|       C002|      P005|2026-08-12|  4500|completed|Bangalore|
|    1008|       C003|      P003|2026-08-12|   900|   FAILED|    Delhi|
|    1001|       C001|      P001|2026-08-10|  2500|Completed|    Delhi|
+--------+-----------+----------+----------+------+---------+---------+



In [ ]:
bronze_products.show()
bronze_clickstream.show()


+-----------+-----+----------+------------+
|   category|price|product_id|product_name|
+-----------+-----+----------+------------+
|Electronics| 2500|      P001|      Laptop|
|Electronics| 1800|      P002|  Headphones|
|    Fashion|  900|      P003|    Backpack|
|  Furniture| 3200|      P004|Office Chair|
|Electronics| 4500|      P005| Smart Watch|
+-----------+-----+----------+------------+

+-----------+--------+-------------------+-----------+----------+
|customer_id|event_id|         event_time| event_type|product_id|
+-----------+--------+-------------------+-----------+----------+
|       C001|    E001|2026-08-10 10:01:00|       view|      P001|
|       C001|    E002|2026-08-10 10:03:00|add_to_cart|      P001|
|       C001|    E003|2026-08-10 10:05:00|   purchase|      P001|
|       C002|    E004|2026-08-10 11:00:00|       view|      P002|
|       C003|    E005|2026-08-11 09:30:00|       view|      P003|
|       C004|    E006|2026-08-11 12:10:00|   purchase|      P004|
|       C

# 10. Data Quality Check on Bronze Orders

Before cleaning data, always inspect common problems.

We will check:

```text
row count
duplicate order_id
missing values
invalid amount
status quality
```


In [ ]:
print("Total bronze orders:", bronze_orders.count())
print("Distinct order IDs:", bronze_orders.select("order_id").distinct().count())
print("Duplicate order count estimate:", bronze_orders.count() - bronze_orders.select("order_id").distinct().count())

bronze_orders.groupBy("status").count().show()


Total bronze orders: 9
Distinct order IDs: 8
Duplicate order count estimate: 1
+---------+-----+
|   status|count|
+---------+-----+
|completed|    2|
|Completed|    4|
|Cancelled|    1|
|   FAILED|    1|
|  Pending|    1|
+---------+-----+



In [ ]:
problem_orders = bronze_orders.filter(
    (col("customer_id").isNull()) |
    (col("amount").isNull()) |
    (col("amount") <= 0)
)

problem_orders.show()


+--------+-----------+----------+----------+------+---------+-------+
|order_id|customer_id|product_id|order_date|amount|   status|   city|
+--------+-----------+----------+----------+------+---------+-------+
|    1002|       C002|      P002|2026-08-10|  -500|completed| Mumbai|
|    1003|       C003|      P003|2026-08-10|  NULL|  Pending|  Delhi|
|    1006|       NULL|      P002|2026-08-11|  1800|Completed|Chennai|
+--------+-----------+----------+----------+------+---------+-------+



# 11. Silver Layer: Clean and Standardize Data

The Silver layer should contain cleaner and more reliable data.

For orders, we will apply these rules:

```text
Remove duplicate order_id
Keep only rows with customer_id
Keep only rows with amount > 0
Standardize status to lowercase
Convert order_date to date
Add processed timestamp
```

We will also create a rejected records dataset.


In [ ]:
silver_orders = (
    bronze_orders
    .dropDuplicates(["order_id"])
    .filter(col("customer_id").isNotNull())
    .filter(col("amount").isNotNull())
    .filter(col("amount") > 0)
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("processed_at", current_timestamp())
)

silver_orders.show()
silver_orders.printSchema()


+--------+-----------+----------+----------+------+---------+---------+--------------------+
|order_id|customer_id|product_id|order_date|amount|   status|     city|        processed_at|
+--------+-----------+----------+----------+------+---------+---------+--------------------+
|    1001|       C001|      P001|2026-08-10|  2500|completed|    Delhi|2026-08-10 12:32:...|
|    1004|       C004|      P004|2026-08-11|  3200|completed|     Pune|2026-08-10 12:32:...|
|    1005|       C005|      P001|2026-08-11|  1200|cancelled|   Mumbai|2026-08-10 12:32:...|
|    1007|       C002|      P005|2026-08-12|  4500|completed|Bangalore|2026-08-10 12:32:...|
|    1008|       C003|      P003|2026-08-12|   900|   failed|    Delhi|2026-08-10 12:32:...|
+--------+-----------+----------+----------+------+---------+---------+--------------------+

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable

In [ ]:
rejected_orders = (
    bronze_orders
    .dropDuplicates(["order_id"])
    .filter(
        (col("customer_id").isNull()) |
        (col("amount").isNull()) |
        (col("amount") <= 0)
    )
    .withColumn("rejection_reason",
        when(col("customer_id").isNull(), lit("missing customer_id"))
        .when(col("amount").isNull(), lit("missing amount"))
        .when(col("amount") <= 0, lit("amount must be greater than zero"))
        .otherwise(lit("unknown"))
    )
)

rejected_orders.show()


+--------+-----------+----------+----------+------+---------+-------+--------------------+
|order_id|customer_id|product_id|order_date|amount|   status|   city|    rejection_reason|
+--------+-----------+----------+----------+------+---------+-------+--------------------+
|    1002|       C002|      P002|2026-08-10|  -500|completed| Mumbai|amount must be gr...|
|    1003|       C003|      P003|2026-08-10|  NULL|  Pending|  Delhi|      missing amount|
|    1006|       NULL|      P002|2026-08-11|  1800|Completed|Chennai| missing customer_id|
+--------+-----------+----------+----------+------+---------+-------+--------------------+



## Clean Customers, Products, Payments, and Clickstream

We will also standardize the other datasets.


In [ ]:
silver_customers = (
    bronze_customers
    .dropDuplicates(["customer_id"])
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("segment", lower(trim(col("segment"))))
    .withColumn("signup_date", to_date(col("signup_date")))
)

silver_products = (
    bronze_products
    .dropDuplicates(["product_id"])
    .withColumn("category", lower(trim(col("category"))))
)

silver_payments = (
    bronze_payments
    .dropDuplicates(["payment_id"])
    .withColumn("payment_status", lower(trim(col("payment_status"))))
    .withColumn("payment_method", lower(trim(col("payment_method"))))
    .withColumn("payment_date", to_date(col("payment_date")))
)

silver_clickstream = (
    bronze_clickstream
    .dropDuplicates(["event_id"])
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("event_time", col("event_time").cast("timestamp"))
)

silver_customers.show()
silver_products.show()
silver_payments.show()
silver_clickstream.show()


+-----------+-------------+--------+-----------+-------+
|customer_id|customer_name| segment|signup_date|country|
+-----------+-------------+--------+-----------+-------+
|       C001| Aarav Sharma| premium| 2025-01-15|  India|
|       C002|   Meera Iyer|standard| 2025-03-20|  India|
|       C003|   Kabir Khan| premium| 2025-05-01|  India|
|       C004|   Ananya Rao|standard| 2025-07-11|  India|
|       C005|  Rohan Gupta|     new| 2026-01-01|  India|
+-----------+-------------+--------+-----------+-------+

+-----------+-----+----------+------------+
|   category|price|product_id|product_name|
+-----------+-----+----------+------------+
|electronics| 2500|      P001|      Laptop|
|electronics| 1800|      P002|  Headphones|
|    fashion|  900|      P003|    Backpack|
|  furniture| 3200|      P004|Office Chair|
|electronics| 4500|      P005| Smart Watch|
+-----------+-----+----------+------------+

+----------+--------+--------------+--------------+------------+
|payment_id|order_id|pay

# 12. Write Silver Data as Parquet

In Data Lakes:

```text
Bronze often stores raw CSV/JSON.
Silver often stores cleaned Parquet.
Gold often stores business-ready Parquet or Delta tables.
```

Parquet is preferred for analytics because it is columnar and compressed.


In [ ]:
silver_orders_output = silver_path / "orders"
silver_customers_output = silver_path / "customers"
silver_products_output = silver_path / "products"
silver_payments_output = silver_path / "payments"
silver_clickstream_output = silver_path / "clickstream_events"
silver_rejected_orders_output = silver_path / "rejected_orders"

silver_orders.write.mode("overwrite").partitionBy("order_date").parquet(str(silver_orders_output))
silver_customers.write.mode("overwrite").parquet(str(silver_customers_output))
silver_products.write.mode("overwrite").parquet(str(silver_products_output))
silver_payments.write.mode("overwrite").partitionBy("payment_date").parquet(str(silver_payments_output))
silver_clickstream.write.mode("overwrite").parquet(str(silver_clickstream_output))
rejected_orders.write.mode("overwrite").parquet(str(silver_rejected_orders_output))

print("Silver layer written successfully.")


Silver layer written successfully.


In [ ]:
for file_path in silver_path.rglob("*"):
    if file_path.is_dir():
        print(file_path)


ecommerce_datalake_demo/data_lake/silver/payments
ecommerce_datalake_demo/data_lake/silver/clickstream_events
ecommerce_datalake_demo/data_lake/silver/orders
ecommerce_datalake_demo/data_lake/silver/rejected_orders
ecommerce_datalake_demo/data_lake/silver/products
ecommerce_datalake_demo/data_lake/silver/customers
ecommerce_datalake_demo/data_lake/silver/payments/payment_date=2026-08-12
ecommerce_datalake_demo/data_lake/silver/payments/payment_date=2026-08-11
ecommerce_datalake_demo/data_lake/silver/payments/payment_date=2026-08-10
ecommerce_datalake_demo/data_lake/silver/orders/order_date=2026-08-10
ecommerce_datalake_demo/data_lake/silver/orders/order_date=2026-08-11
ecommerce_datalake_demo/data_lake/silver/orders/order_date=2026-08-12


# 13. Gold Layer: Business-Ready Analytics

Gold data answers business questions.

We will create:

```text
1. revenue_by_city
2. revenue_by_product_category
3. customer_order_summary
4. payment_success_report
5. clickstream_event_summary
```

```mermaid
flowchart LR
    SO[Silver Orders] --> G1[Gold Revenue by City]
    SO --> Join1[Join Orders + Products]
    SP[Silver Products] --> Join1
    Join1 --> G2[Gold Revenue by Category]

    SO --> Join2[Join Orders + Customers]
    SC[Silver Customers] --> Join2
    Join2 --> G3[Gold Customer Summary]

    PAY[Silver Payments] --> G4[Gold Payment Success Report]
    CLK[Silver Clickstream] --> G5[Gold Clickstream Summary]
```


## 13.1 Gold Table: Revenue by City


In [ ]:
gold_revenue_by_city = (
    silver_orders
    .filter(col("status") == "completed")
    .groupBy("city")
    .agg(
        spark_sum("amount").alias("total_revenue"),
        count("*").alias("completed_order_count")
    )
    .orderBy(desc("total_revenue"))
)

gold_revenue_by_city.show()


+---------+-------------+---------------------+
|     city|total_revenue|completed_order_count|
+---------+-------------+---------------------+
|Bangalore|         4500|                    1|
|     Pune|         3200|                    1|
|    Delhi|         2500|                    1|
+---------+-------------+---------------------+



## 13.2 Gold Table: Revenue by Product Category


In [ ]:
orders_products = (
    silver_orders.alias("o")
    .join(silver_products.alias("p"), col("o.product_id") == col("p.product_id"), "left")
)

gold_revenue_by_category = (
    orders_products
    .filter(col("status") == "completed")
    .groupBy("category")
    .agg(
        spark_sum("amount").alias("total_revenue"),
        count("*").alias("completed_order_count")
    )
    .orderBy(desc("total_revenue"))
)

gold_revenue_by_category.show()


+-----------+-------------+---------------------+
|   category|total_revenue|completed_order_count|
+-----------+-------------+---------------------+
|electronics|         7000|                    2|
|  furniture|         3200|                    1|
+-----------+-------------+---------------------+



## 13.3 Gold Table: Customer Order Summary


In [ ]:
orders_customers = (
    silver_orders.alias("o")
    .join(silver_customers.alias("c"), col("o.customer_id") == col("c.customer_id"), "left")
)

gold_customer_summary = (
    orders_customers
    .groupBy("o.customer_id", "customer_name", "segment")
    .agg(
        count("*").alias("total_orders"),
        spark_sum("amount").alias("total_order_amount"),
        avg("amount").alias("avg_order_amount")
    )
    .orderBy(desc("total_order_amount"))
)

gold_customer_summary.show()


+-----------+-------------+--------+------------+------------------+----------------+
|customer_id|customer_name| segment|total_orders|total_order_amount|avg_order_amount|
+-----------+-------------+--------+------------+------------------+----------------+
|       C002|   Meera Iyer|standard|           1|              4500|          4500.0|
|       C004|   Ananya Rao|standard|           1|              3200|          3200.0|
|       C001| Aarav Sharma| premium|           1|              2500|          2500.0|
|       C005|  Rohan Gupta|     new|           1|              1200|          1200.0|
|       C003|   Kabir Khan| premium|           1|               900|           900.0|
+-----------+-------------+--------+------------+------------------+----------------+



## 13.4 Gold Table: Payment Success Report


In [ ]:
gold_payment_success_report = (
    silver_payments
    .groupBy("payment_status")
    .agg(count("*").alias("payment_count"))
    .orderBy(desc("payment_count"))
)

gold_payment_success_report.show()


+--------------+-------------+
|payment_status|payment_count|
+--------------+-------------+
|       success|            3|
|        failed|            2|
+--------------+-------------+



## 13.5 Gold Table: Clickstream Event Summary


In [ ]:
gold_clickstream_summary = (
    silver_clickstream
    .groupBy("event_type")
    .agg(count("*").alias("event_count"))
    .orderBy(desc("event_count"))
)

gold_clickstream_summary.show()


+-----------+-----------+
| event_type|event_count|
+-----------+-----------+
|   purchase|          3|
|       view|          3|
|add_to_cart|          1|
+-----------+-----------+



# 14. Write Gold Layer as Parquet

Gold layer is usually consumed by:

```text
BI dashboards
analytics teams
data science teams
reporting jobs
warehouse loads
```


In [ ]:
gold_revenue_by_city.write.mode("overwrite").parquet(str(gold_path / "revenue_by_city"))
gold_revenue_by_category.write.mode("overwrite").parquet(str(gold_path / "revenue_by_category"))
gold_customer_summary.write.mode("overwrite").parquet(str(gold_path / "customer_summary"))
gold_payment_success_report.write.mode("overwrite").parquet(str(gold_path / "payment_success_report"))
gold_clickstream_summary.write.mode("overwrite").parquet(str(gold_path / "clickstream_event_summary"))

print("Gold layer written successfully.")


Gold layer written successfully.


In [ ]:
for path in data_lake_path.rglob("*"):
    if path.is_dir():
        print(path)


ecommerce_datalake_demo/data_lake/gold
ecommerce_datalake_demo/data_lake/bronze
ecommerce_datalake_demo/data_lake/silver
ecommerce_datalake_demo/data_lake/gold/revenue_by_category
ecommerce_datalake_demo/data_lake/gold/customer_summary
ecommerce_datalake_demo/data_lake/gold/revenue_by_city
ecommerce_datalake_demo/data_lake/gold/clickstream_event_summary
ecommerce_datalake_demo/data_lake/gold/payment_success_report
ecommerce_datalake_demo/data_lake/bronze/payments
ecommerce_datalake_demo/data_lake/bronze/clickstream_events
ecommerce_datalake_demo/data_lake/bronze/orders
ecommerce_datalake_demo/data_lake/bronze/products
ecommerce_datalake_demo/data_lake/bronze/customers
ecommerce_datalake_demo/data_lake/silver/payments
ecommerce_datalake_demo/data_lake/silver/clickstream_events
ecommerce_datalake_demo/data_lake/silver/orders
ecommerce_datalake_demo/data_lake/silver/rejected_orders
ecommerce_datalake_demo/data_lake/silver/products
ecommerce_datalake_demo/data_lake/silver/customers
ecommer

# 15. Read Gold Data Back

A downstream analytics team may directly query the Gold layer.

Let us read one Gold table.


In [ ]:
revenue_city_from_lake = spark.read.parquet(str(gold_path / "revenue_by_city"))
revenue_city_from_lake.show()


+---------+-------------+---------------------+
|     city|total_revenue|completed_order_count|
+---------+-------------+---------------------+
|Bangalore|         4500|                    1|
|     Pune|         3200|                    1|
|    Delhi|         2500|                    1|
+---------+-------------+---------------------+



# 16. Data Lake File Format Strategy

A practical strategy:

| Layer | Common Format | Reason |
|---|---|---|
| Bronze | CSV/JSON | Preserve source data as received |
| Silver | Parquet | Clean, compressed, efficient analytics format |
| Gold | Parquet or Delta | Business-ready and query optimized |

## Simple rule

```text
Keep Bronze close to raw.
Make Silver clean and standardized.
Make Gold business-ready.
```


# 17. Partitioning Strategy

Partitioning stores data in folders based on column values.

Example:

```text
silver/orders/order_date=2026-08-10/
silver/orders/order_date=2026-08-11/
```

This helps Spark read only relevant folders when filtering by date.

```mermaid
flowchart TD
    A[silver/orders] --> D1[order_date=2026-08-10]
    A --> D2[order_date=2026-08-11]
    A --> D3[order_date=2026-08-12]
```

## Good partition columns

```text
date
country
region
event_type
business_unit
```

## Bad partition columns

```text
order_id
customer_id
email
transaction_id
```

Bad partition columns create too many small folders.


# 18. AWS S3 Data Lake Hands-On

Now we will create the same Data Lake structure on AWS S3.

## AWS architecture

```mermaid
flowchart LR
    Local[Local/Colab Files] --> Boto3[boto3 Upload]
    Boto3 --> S3Bronze[S3 Bronze Layer]
    S3Bronze --> Glue[AWS Glue / Spark / EMR / Databricks]
    Glue --> S3Silver[S3 Silver Layer]
    S3Silver --> S3Gold[S3 Gold Layer]
    S3Gold --> Athena[Athena / BI / Snowflake / Databricks SQL]
```

## What students need

To run the AWS section, students need:

```text
AWS account or AWS Educate lab access
An S3 bucket
IAM user or role with S3 permissions
AWS_ACCESS_KEY_ID
AWS_SECRET_ACCESS_KEY
AWS_DEFAULT_REGION
```

## Important safety note

Do not hardcode AWS keys directly in notebooks.

Use Colab Secrets or environment variables.


# 19. Install boto3 for AWS S3

Run this section only if you have AWS access.

If you do not have AWS access today, you can still understand the structure and run the local Data Lake sections above.


In [ ]:
!pip install boto3 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.4 MB/s eta 0:00:00


# 20. Configure AWS Credentials Safely

## Recommended for Google Colab

Use Colab Secrets:

```text
AWS_ACCESS_KEY_ID
AWS_SECRET_ACCESS_KEY
AWS_DEFAULT_REGION
```

Then this cell will read them securely.

If you are not using Colab Secrets, you can set environment variables manually for a temporary classroom demo.


In [ ]:
import os

try:
    from google.colab import userdata

    aws_access_key = userdata.get("AWS_ACCESS_KEY_ID")
    aws_secret_key = userdata.get("AWS_SECRET_ACCESS_KEY")
    aws_region = userdata.get("AWS_DEFAULT_REGION") or "ap-south-1"

    if aws_access_key and aws_secret_key:
        os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key
        os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_key
        os.environ["AWS_DEFAULT_REGION"] = aws_region
        print("AWS credentials loaded from Colab Secrets.")
    else:
        print("AWS credentials not found in Colab Secrets.")
except Exception:
    print("Not running in Colab or Colab Secrets unavailable.")


Not running in Colab or Colab Secrets unavailable.


# 21. Set Your S3 Bucket Name

Use an existing bucket or create one from the AWS Console.

Bucket names must be globally unique.

Example:

```text
manan-ecommerce-datalake-demo-2026
```

Change the value below before running.


In [ ]:
S3_BUCKET = "replace-with-your-s3-bucket-name"
S3_PREFIX = "ecommerce-datalake"

print("Target S3 path:")
print(f"s3://{S3_BUCKET}/{S3_PREFIX}/")


Target S3 path:
s3://replace-with-your-s3-bucket-name/ecommerce-datalake/


# 22. Create S3 Data Lake Prefixes

S3 is object storage, not a traditional folder filesystem.

But we organize files using prefixes like:

```text
ecommerce-datalake/bronze/
ecommerce-datalake/silver/
ecommerce-datalake/gold/
```


In [ ]:
import boto3


def get_s3_client():
    return boto3.client("s3")


def create_s3_prefixes(bucket, prefix):
    s3 = get_s3_client()

    prefixes = [
        f"{prefix}/bronze/",
        f"{prefix}/silver/",
        f"{prefix}/gold/"
    ]

    for p in prefixes:
        s3.put_object(Bucket=bucket, Key=p)

    print("Created S3 prefixes:")
    for p in prefixes:
        print(f"s3://{bucket}/{p}")

# Uncomment after setting S3_BUCKET
# create_s3_prefixes(S3_BUCKET, S3_PREFIX)


# 23. Upload Bronze Files to S3

This uploads the raw source files to the S3 Bronze layer.

Expected S3 paths:

```text
s3://bucket/ecommerce-datalake/bronze/orders/ingestion_date=2026-08-10/orders.csv
s3://bucket/ecommerce-datalake/bronze/customers/ingestion_date=2026-08-10/customers.csv
s3://bucket/ecommerce-datalake/bronze/products/ingestion_date=2026-08-10/products.json
s3://bucket/ecommerce-datalake/bronze/payments/ingestion_date=2026-08-10/payments.csv
s3://bucket/ecommerce-datalake/bronze/clickstream_events/ingestion_date=2026-08-10/clickstream_events.json
```


In [ ]:
def upload_file_to_s3(local_file, bucket, s3_key):
    s3 = get_s3_client()
    s3.upload_file(str(local_file), bucket, s3_key)
    print(f"Uploaded: s3://{bucket}/{s3_key}")


def upload_bronze_files_to_s3(bucket, prefix, ingestion_date):
    files_to_upload = [
        (orders_file, f"{prefix}/bronze/orders/ingestion_date={ingestion_date}/orders.csv"),
        (customers_file, f"{prefix}/bronze/customers/ingestion_date={ingestion_date}/customers.csv"),
        (products_file, f"{prefix}/bronze/products/ingestion_date={ingestion_date}/products.json"),
        (payments_file, f"{prefix}/bronze/payments/ingestion_date={ingestion_date}/payments.csv"),
        (clickstream_file, f"{prefix}/bronze/clickstream_events/ingestion_date={ingestion_date}/clickstream_events.json"),
    ]

    for local_file, s3_key in files_to_upload:
        upload_file_to_s3(local_file, bucket, s3_key)

# Uncomment after setting S3_BUCKET
# upload_bronze_files_to_s3(S3_BUCKET, S3_PREFIX, ingestion_date)


# 24. Upload Silver and Gold Data Lake Outputs to S3

This section uploads your processed Parquet outputs to S3.

Because Spark writes folders containing multiple part files, we need to upload a directory recursively.


In [ ]:
def upload_directory_to_s3(local_dir, bucket, s3_prefix):
    s3 = get_s3_client()
    local_dir = Path(local_dir)

    for file_path in local_dir.rglob("*"):
        if file_path.is_file():
            relative_path = file_path.relative_to(local_dir)
            s3_key = f"{s3_prefix}/{relative_path}".replace("\\", "/")
            s3.upload_file(str(file_path), bucket, s3_key)

    print(f"Uploaded directory {local_dir} to s3://{bucket}/{s3_prefix}")


def upload_processed_layers_to_s3(bucket, prefix):
    upload_directory_to_s3(silver_path, bucket, f"{prefix}/silver")
    upload_directory_to_s3(gold_path, bucket, f"{prefix}/gold")

# Uncomment after setting S3_BUCKET and running previous layers
# upload_processed_layers_to_s3(S3_BUCKET, S3_PREFIX)


# 25. List S3 Data Lake Objects

Use this to verify your S3 Data Lake structure.


In [ ]:
def list_s3_objects(bucket, prefix, max_keys=50):
    s3 = get_s3_client()

    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
        MaxKeys=max_keys
    )

    contents = response.get("Contents", [])

    if not contents:
        print("No objects found.")
        return

    for obj in contents:
        print(f"s3://{bucket}/{obj['Key']}")

# Uncomment after uploading
# list_s3_objects(S3_BUCKET, S3_PREFIX, max_keys=100)


# 26. AWS Data Lake Checkpoint

At this point, students should be able to show:

```text
1. S3 bucket name
2. bronze/ prefix with raw CSV and JSON files
3. silver/ prefix with cleaned Parquet files
4. gold/ prefix with business-ready Parquet files
5. screenshot of S3 folder structure
```

## Reflection questions

Answer these:

```text
Why do we keep raw files in Bronze?
Why do we write Silver as Parquet?
Why do we create Gold tables?
What happens if the Bronze layer is deleted?
Which layer should BI dashboards usually query?
```


# 27. How This Connects to Delta Lake and Databricks

Plain Data Lakes are powerful, but they have limitations:

```text
No built-in ACID transactions
No easy time travel
No built-in schema enforcement
Hard to safely update and delete records
Pipeline failures can leave inconsistent files
```

This leads to the next topic:

```text
Delta Lake
```

Delta Lake makes Data Lake files behave more like reliable tables.

Then Databricks provides a managed platform for:

```text
Spark
Delta Lake
Notebooks
Workflows
SQL
Governance
Lakehouse pipelines
```

```mermaid
flowchart LR
    DL[Data Lake<br/>Files on S3/HDFS/ADLS] --> Delta[Delta Lake<br/>Reliable Tables]
    Delta --> DBX[Databricks<br/>Managed Lakehouse Platform]
    DBX --> SQL[SQL Analytics]
    DBX --> Jobs[Workflows]
    DBX --> ML[ML/AI]
```


# 28. Student Assignment

## Assignment: Build Your Own AWS Data Lake

Use the same source files:

```text
orders.csv
customers.csv
products.json
payments.csv
clickstream_events.json
```

## Tasks

1. Create a local Data Lake with Bronze, Silver, and Gold layers.
2. Store raw files in Bronze.
3. Clean and standardize data into Silver.
4. Create at least three Gold business reports:
   - revenue by city
   - revenue by product category
   - customer order summary
5. Upload the same structure to AWS S3.
6. Submit screenshots and short explanations.

## Required submission

```text
Notebook
S3 bucket screenshot
Bronze/Silver/Gold folder screenshot
Gold report outputs
Short explanation of Data Lake layers
```


# 29. Interview Questions

## Q1. What is a Data Lake?

A Data Lake is a scalable storage system used to store raw and processed data files for analytics, data engineering, and machine learning.

## Q2. What are Bronze, Silver, and Gold layers?

Bronze stores raw source data. Silver stores cleaned and standardized data. Gold stores business-ready analytics data.

## Q3. Why is Parquet commonly used in Data Lakes?

Parquet is columnar, compressed, and efficient for analytical queries because tools like Spark can read only required columns.

## Q4. Why is AWS S3 commonly used for Data Lakes?

S3 is scalable object storage that can store large volumes of files and can be used by Spark, Glue, Athena, EMR, Databricks, and other analytics tools.

## Q5. What is a Data Swamp?

A Data Swamp is a poorly organized Data Lake with unclear ownership, messy folder structures, missing quality checks, and low trust.

## Q6. Why do we need Delta Lake after learning Data Lakes?

Delta Lake adds reliability features like transactions, schema enforcement, history, and time travel on top of Data Lake storage.


# 30. Final Summary

In this notebook, you built a practical Data Lake.

You created:

```text
Source files
Bronze raw layer
Silver cleaned layer
Gold analytics layer
Optional AWS S3 Data Lake
```

Main takeaway:

```text
A Data Lake is not just a storage folder.
A good Data Lake is organized into layers, uses efficient file formats, applies data quality rules, and prepares data for analytics and downstream platforms.
```

Next topic:

```text
Delta Lake: Making Data Lakes Reliable
```
